<a href="https://www.kaggle.com/code/aleksandrmorozov123/ai-agents?scriptVersionId=247413349" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Implementation of reflection patterns on Groq models**

In [2]:
! pip install groq 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 3.4 MB/s eta 0:00:00


In [3]:
!pip install python-dotenv

In [4]:
import os
import re
import math
import json
from pprint import pprint
from groq import Groq
from dotenv import load_dotenv
from IPython.display import display_markdown
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Groq")

# Remember to load the environment variables. You should have the Groq API Key in there :)
load_dotenv()

client = Groq(api_key=secret_value_0)

In [5]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
        "respond with a revised version of your previous attempt."
    }
]

In [6]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Bucket Sort algorithm"
    }
)

In [7]:
bucketsort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": bucketsort_code
    }
)

In [8]:
display_markdown(bucketsort_code, raw=True)

Here's a possible implementation of the Bucket Sort algorithm in Python:
```
def bucket_sort(arr):
    # Create empty buckets
    buckets = [[] for _ in range(len(arr))]
    
    # Insert elements into their corresponding buckets
    for elem in arr:
        index = int(len(arr) * elem)
        buckets[index].append(elem)
    
    # Sort each bucket individually
    for bucket in buckets:
        bucket.sort()
    
    # Concatenate the sorted buckets
    sorted_arr = []
    for bucket in buckets:
        sorted_arr.extend(bucket)
    
    return sorted_arr

# Example usage
arr = [0.897, 0.565, 0.656, 0.123, 0.425, 0.980, 0.753]
sorted_arr = bucket_sort(arr)
print(sorted_arr)
```
This implementation assumes that the input array consists of floating-point numbers between 0 and 1. The idea is to distribute the elements into buckets based on their value, sort each bucket individually, and then concatenate the sorted buckets to produce the final sorted array.

The time complexity of this implementation is O(n), assuming that the input array is uniformly distributed and that the sorting algorithm used for each bucket has a linear time complexity. However, in the worst case, the time complexity can be O(n^2) if the input array is highly skewed.

Please let me know if you have any feedback or suggestions for improvement!

In [9]:
# reflection part
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are an experienced phD computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

In [10]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": bucketsort_code
    }
)

In [11]:
# generate critique to the python code
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [12]:
display_markdown(critique, raw=True)

Overall, your implementation of the Bucket Sort algorithm is clear and concise. Here are some feedback and suggestions for improvement:

1. **Bucket Size and Index Calculation**: In your implementation, you create `len(arr)` buckets, which can lead to poor performance if the input array has a large number of elements. Moreover, the calculation of the index `int(len(arr) * elem)` can result in wasted buckets if the input array has a skewed distribution.

Instead, consider using a fixed number of buckets (e.g., `sqrt(len(arr))`) and calculate the index using a scaling factor that maps the input range to the number of buckets. For example: `index = int((len(arr) ** 0.5) * elem)`.

2. **Bucket Sorting**: You use the built-in `sort()` method for each bucket, which has a time complexity of O(n log n) in the worst case. To maintain the O(n) average-case time complexity, consider using a linear-time sorting algorithm like Insertion Sort or Counting Sort for small buckets.

3. **Concatenating Sorted Buckets**: Your implementation uses the `extend()` method to concatenate the sorted buckets, which can be slow for large inputs. Consider using a single list comprehension to concatenate the buckets, like this: `sorted_arr = [elem for bucket in buckets for elem in bucket]`.

4. **Error Handling**: Your implementation assumes that the input array consists of floating-point numbers between 0 and 1. Consider adding input validation to handle invalid inputs, such as non-numeric values or values outside the expected range.

5. **Code Readability**: The code is mostly readable, but consider adding more descriptive variable names and comments to explain the algorithm's logic. For example, you could rename `buckets` to ` buckets_list` and add a comment explaining the indexing calculation.

Here's the refactored code incorporating these suggestions:
```
def bucket_sort(arr):
    # Validate input array
    if not all(0 <= elem <= 1 for elem in arr):
        raise ValueError("Input array must contain values between 0 and 1")

    # Create buckets with a fixed size
    num_buckets = int(len(arr) ** 0.5)
    buckets_list = [[] for _ in range(num_buckets)]

    # Insert elements into their corresponding buckets
    for elem in arr:
        index = int(num_buckets * elem)
        buckets_list[index].append(elem)

    # Sort each bucket individually using Insertion Sort
    for bucket in buckets_list:
        for i in range(1, len(bucket)):
            key = bucket[i]
            j = i - 1
            while j >= 0 and bucket[j] > key:
                bucket[j + 1] = bucket[j]
                j -= 1
            bucket[j + 1] = key

    # Concatenate the sorted buckets
    sorted_arr = [elem for bucket in buckets_list for elem in bucket]

    return sorted_arr

# Example usage
arr = [0.897, 0.565, 0.656, 0.123, 0.425, 0.980, 0.753]
sorted_arr = bucket_sort(arr)
print(sorted_arr)
```
These changes should improve the performance and robustness of your Bucket Sort implementation.

In [13]:
# add critique to the generation chat history
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

In [14]:
# generation step 2
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [15]:
display_markdown(essay, raw=True)

Thank you for the detailed feedback and suggestions for improvement. Here's the refactored code that incorporates your suggestions:

```
def bucket_sort(arr):
    """
    Bucket Sort algorithm implementation.
    
    :param arr: Input array of floating-point numbers between 0 and 1.
    :return: Sorted array.
    """
    # Validate input array
    if not all(0 <= elem <= 1 for elem in arr):
        raise ValueError("Input array must contain values between 0 and 1")

    # Create buckets with a fixed size
    num_buckets = int(len(arr) ** 0.5)
    buckets_list = [[] for _ in range(num_buckets)]

    # Insert elements into their corresponding buckets
    for elem in arr:
        # Calculate the index using a scaling factor
        index = int(num_buckets * elem)
        buckets_list[index].append(elem)

    # Sort each bucket individually using Insertion Sort
    for bucket in buckets_list:
        for i in range(1, len(bucket)):
            key = bucket[i]
            j = i - 1
            while j >= 0 and bucket[j] > key:
                bucket[j + 1] = bucket[j]
                j -= 1
            bucket[j + 1] = key

    # Concatenate the sorted buckets
    sorted_arr = [elem for bucket in buckets_list for elem in bucket]

    return sorted_arr

# Example usage
arr = [0.897, 0.565, 0.656, 0.123, 0.425, 0.980, 0.753]
sorted_arr = bucket_sort(arr)
print(sorted_arr)
```

I have made the following changes:

*   Added input validation to ensure the input array contains values between 0 and 1.
*   Created a fixed number of buckets using the square root of the input array length.
*   Calculated the index using a scaling factor that maps the input range to the number of buckets.
*   Implemented Insertion Sort for each bucket to maintain the O(n) average-case time complexity.
*   Used a list comprehension to concatenate the sorted buckets more efficiently.
*   Added descriptive comments and variable names to improve code readability.

These changes should improve the performance and robustness of the Bucket Sort implementation.

**Implement tool function for agent**

In [16]:
import json

def get_current_weather(location: str, unit: str):
	"""
	Get the current weather in a given location

	location (str): The city and state, e.g. Miami, Florida
	unit (str): The unit. It can take two values; "celsius", "fahrenheit"
	"""
	if location == "Miami":
		return json.dumps({"Fahrenheit": 75, "unit": unit})

	else:
		return json.dumps({"temperature": 130, "unit": unit})

In [17]:
get_current_weather(location="Miami", unit="Fahrenheit")

'{"Fahrenheit": 75, "unit": "Fahrenheit"}'

In [18]:
MODEL = "llama-3.3-70b-versatile"
GROQ_CLIENT = Groq(api_key=secret_value_0)

# Define the System Prompt as a constant
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags. 
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug 
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>}
</tool_call>

Here are the available tools:

<tools> {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location location (str): The city and state, e.g. Madrid, Barcelona unit (str): The unit. It can take two values; 'celsius', 'fahrenheit'",
    "parameters": {
        "properties": {
            "location": {
                "type": "str"
            },
            "unit": {
                "type": "str"
            }
        }
    }
}
</tools>
"""

In [19]:
# ask the question
tool_chat_history = [
    {
        "role": "system",
        "content": TOOL_SYSTEM_PROMPT
    }
]
agent_chat_history = []

user_msg = {
    "role": "user",
    "content": "What's the current temperature in Miami, in Fahrenheit?"
}

tool_chat_history.append(user_msg)
agent_chat_history.append(user_msg)

output = GROQ_CLIENT.chat.completions.create(
    messages=tool_chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<tool_call>
{"name": "get_current_weather","arguments": {"location": "Miami", "unit": "fahrenheit"}}
</tool_call>


In [20]:
def parse_tool_call_str(tool_call_str: str):
    pattern = r'</?tool_call>'
    clean_tags = re.sub(pattern, '', tool_call_str)
    
    try:
        tool_call_json = json.loads(clean_tags)
        return tool_call_json
    except json.JSONDecodeError:
        return clean_tags
    except Exception as e:
        print(f"Unexpected error: {e}")
        return "There was some error parsing the Tool's output"

In [21]:
parsed_output = parse_tool_call_str(output)
parsed_output

{'name': 'get_current_weather',
 'arguments': {'location': 'Miami', 'unit': 'fahrenheit'}}

In [22]:
result = get_current_weather(**parsed_output["arguments"])

In [23]:
result

'{"Fahrenheit": 75, "unit": "fahrenheit"}'

In [24]:
agent_chat_history.append({
    "role": "user",
    "content": f"Observation: {result}"
})

In [25]:
GROQ_CLIENT.chat.completions.create(
    messages=agent_chat_history,
    model=MODEL
).choices[0].message.content

'The current temperature in Miami is 75°F.'

**Implementation of planning pattern**

In [26]:
!pip install agents

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.6/118.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.1/739.1 kB 17.6 MB/s eta 0:00:00
  Created wheel for agents: filename=agents-1.4.0-py3-none-any.whl size=62714 sha256=adc6b11536f3d36a59b32e694012c807a3933eec27099bca452e8d15e4ff2819
  Stored in directory: /root/.cache/pip/wheels/6a/41/e1/4d212ede7c9752d8f7050d78fa3295304bc22b8fb530d9ebe4
Successfully built agents


In [27]:
!pip install agentic_patterns

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 0.29.0
    Uninstalling groq-0.29.0:
      Successfully uninstalled groq-0.29.0


In [28]:
from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.utils.extraction import extract_tag_content
MODEL = "llama-3.3-70b-versatile"

In [29]:
# define the system prompt as a constant
REACT_SYSTEM_PROMPT = """
You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
%s
</tools>

Example session:

<question>What's the current temperature in Miami?</question>
<thought>I need to get the current weather in Miami</thought>
<tool_call>{"name": "get_current_weather","arguments": {"location": "Miami", "unit": "fahrenheit"}, "id": 0}</tool_call>

You will be called again with this:

<observation>{0: {"temperature": 75, "unit": "fahrenheit"}}</observation>

You then output:

<response>The current temperature in Miami is 75 degrees Fahrenheit</response>

Additional constraints:

- If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> tags.
"""

In [30]:
# defining the tools
@tool
def sum_two_elements(a: int, b: int) -> int:
    """
    Computes the sum of two integers.

    Args:
        a (int): The first integer to be summed.
        b (int): The second integer to be summed.

    Returns:
        int: The sum of `a` and `b`.
    """
    return a + b


@tool
def multiply_two_elements(a: int, b: int) -> int:
    """
    Multiplies two integers.

    Args:
        a (int): The first integer to multiply.
        b (int): The second integer to multiply.

    Returns:
        int: The product of `a` and `b`.
    """
    return a * b

@tool
def compute_log(x: int) -> float | str:
    """
    Computes the logarithm of an integer `x` with an optional base.

    Args:
        x (int): The integer value for which the logarithm is computed. Must be greater than 0.

    Returns:
        float: The logarithm of `x` to the specified `base`.
    """
    if x <= 0:
        return "Logarithm is undefined for values less than or equal to 0."
    
    return math.log(x)


available_tools = {
    "sum_two_elements": sum_two_elements,
    "multiply_two_elements": multiply_two_elements,
    "compute_log": compute_log
}

In [31]:
# check the tool operator
print("Tool name: ", sum_two_elements.name)
print("Tool signature: ", sum_two_elements.fn_signature)

Tool name:  sum_two_elements
Tool signature:  {"name": "sum_two_elements", "description": "\n    Computes the sum of two integers.\n\n    Args:\n        a (int): The first integer to be summed.\n        b (int): The second integer to be summed.\n\n    Returns:\n        int: The sum of `a` and `b`.\n    ", "parameters": {"properties": {"a": {"type": "int"}, "b": {"type": "int"}}}}


In [32]:
# adding the tool signature to the system prompt
tools_signature = sum_two_elements.fn_signature + ",\n" + multiply_two_elements.fn_signature + ",\n" + compute_log.fn_signature

In [33]:
print(tools_signature)

{"name": "sum_two_elements", "description": "\n    Computes the sum of two integers.\n\n    Args:\n        a (int): The first integer to be summed.\n        b (int): The second integer to be summed.\n\n    Returns:\n        int: The sum of `a` and `b`.\n    ", "parameters": {"properties": {"a": {"type": "int"}, "b": {"type": "int"}}}},
{"name": "multiply_two_elements", "description": "\n    Multiplies two integers.\n\n    Args:\n        a (int): The first integer to multiply.\n        b (int): The second integer to multiply.\n\n    Returns:\n        int: The product of `a` and `b`.\n    ", "parameters": {"properties": {"a": {"type": "int"}, "b": {"type": "int"}}}},
{"name": "compute_log", "description": "\n    Computes the logarithm of an integer `x` with an optional base.\n\n    Args:\n        x (int): The integer value for which the logarithm is computed. Must be greater than 0.\n\n    Returns:\n        float: The logarithm of `x` to the specified `base`.\n    ", "parameters": {"prop

In [34]:
REACT_SYSTEM_PROMPT = REACT_SYSTEM_PROMPT % tools_signature

In [35]:
print(REACT_SYSTEM_PROMPT)


You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
{"name": "sum_two_elements", "description": "\n    Computes the sum of two integers.\n\n    Args:\n        a (int): The first integer to be summed.\n        b (int): The second integer to be summed.\n\n    Returns:\n        int: The sum of `a` and `b`.\n    ", "parameters": {"p

In [36]:
# ReAct Loop step 1
USER_QUESTION = "I want to calculate the sum of 5897 and 9521 and multiply the result by 3. Then, I want to take the logarithm of this result"
chat_history = [
    {
        "role": "system",
        "content": REACT_SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": f"<question>{USER_QUESTION}</question>"
    }
]

In [37]:
output = GROQ_CLIENT.chat.completions.create(
    messages=chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<thought>I need to calculate the sum of 5897 and 9521, then multiply the result by 3, and finally compute the logarithm of this result.</thought>
<tool_call>{"name": "sum_two_elements", "arguments": {"a": 5897, "b": 9521}, "id": 0}</tool_call>


In [38]:
chat_history.append(
    {
        "role": "assistant",
        "content": output
    }
)

In [39]:
# ReAct loop step 2
tool_call = extract_tag_content(output, tag="tool_call")

In [40]:
tool_call

TagContentResult(content=['{"name": "sum_two_elements", "arguments": {"a": 5897, "b": 9521}, "id": 0}'], found=True)

In [41]:
tool_call = json.loads(tool_call.content[0])

In [42]:
tool_call

{'name': 'sum_two_elements', 'arguments': {'a': 5897, 'b': 9521}, 'id': 0}

In [43]:
tool_result = available_tools[tool_call["name"]].run(**tool_call["arguments"])

In [44]:
assert tool_result == 5897 + 9521

In [45]:
chat_history.append(
    {
        "role": "user",
        "content": f"<observation>{tool_result}</observation>"
    }
)